# 🎚️ DJIA — Track Pairing for Mixing

Rank the **best pairs of tracks to mix** from your analyzed library, so you know what to load
next to what on the Hercules.

A pair scores high when the two tracks are **beat-, harmonic-, mood-, and energy-compatible**:

| Signal | Weight | Why it matters for mixing |
|---|---|---|
| **BPM** | 35% | How much you'd have to pitch one track to beatmatch |
| **Key (Camelot)** | 30% | Harmonic mixing — avoids clashing melodies |
| **Mood** | 20% | Keeps the vibe continuous across the blend |
| **Energy** | 15% | Smooth energy arc, no jarring jump |

> **Prereq:** your library must be analyzed into `data/djia.db` first:
> `python -m src.cli analyze data/`

The scoring is **symmetric**, so pairs are unordered; a `play_first` hint tells you which one
to open with (lower energy first = build up).

## 1. Setup & parameters

In [ ]:
import os, sys, sqlite3, re, itertools
import numpy as np
import pandas as pd

# Make `src` importable when running from the repo root
REPO = os.path.abspath('.')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.ai.transition_mapper import (
    _calculate_bpm_compatibility,
    _calculate_mood_continuity,
    _calculate_energy_arc,
)

# ---- Tunables -------------------------------------------------------------
DB_PATH       = 'data/djia.db'   # analyzed library
BPM_TOLERANCE = 6.0              # drop pairs further apart than this (BPM). None = no filter
MIN_SCORE     = 0.0             # only keep pairs at/above this overall score
TOP_N         = 25              # how many pairs to show
WEIGHTS       = {'bpm': 0.35, 'key': 0.30, 'mood': 0.20, 'energy': 0.15}
MOOD_DIMS     = ['dark', 'hypnotic', 'euphoric', 'aggressive', 'industrial', 'minimal']
print('Setup OK — DB:', DB_PATH, '| exists:', os.path.exists(DB_PATH))

## 2. Load the analyzed library

In [ ]:
def load_library(db_path=DB_PATH):
    """One row per track: metadata + features + mood (+ key column if it exists)."""
    if not os.path.exists(db_path):
        raise FileNotFoundError(f'No DB at {db_path}. Run: python -m src.cli analyze data/')
    conn = sqlite3.connect(db_path); conn.row_factory = sqlite3.Row
    try:
        # Detect whether a key column was ever added to features (not in the stock schema)
        fcols = {r['name'] for r in conn.execute('PRAGMA table_info(features)')}
        key_expr = 'f.camelot_key' if 'camelot_key' in fcols else ('f.key' if 'key' in fcols else 'NULL')
        rows = conn.execute(f'''
            SELECT t.id, t.title, t.artist, t.file_name, t.duration,
                   f.bpm, f.rms_mean, f.spectral_centroid_mean, f.spectral_rolloff_mean,
                   f.harmonic_ratio, f.percussive_ratio, f.chroma_variance, f.chroma_entropy,
                   {key_expr} AS camelot_key,
                   m.dark, m.hypnotic, m.euphoric, m.aggressive, m.industrial, m.minimal
            FROM tracks t
            LEFT JOIN features f ON t.id = f.track_id
            LEFT JOIN mood     m ON t.id = m.track_id
            ORDER BY t.id
        ''').fetchall()
    finally:
        conn.close()
    df = pd.DataFrame([dict(r) for r in rows])
    return df

lib = load_library()
n_total = len(lib)
n_ready = int(lib['bpm'].notna().sum()) if n_total else 0
print(f'{n_total} tracks in DB, {n_ready} with BPM/features.')
if n_ready < 2:
    print('\n⚠️  Need at least 2 fully-analyzed tracks to form pairs.')
    print('    Run:  python -m src.cli analyze data/')
lib.head(10)

## 3. (Optional) Enrich with musical key

**Heads up:** the stock DB schema does *not* store musical key, so harmonic scoring below will be
**neutral (0.5)** for every pair unless a `camelot_key` column is present.

This cell computes the Camelot key **live from the audio files** and adds it to `lib`.
It's off by default because it loads every track with librosa (slow).
Set `ENRICH_KEYS = True` to run it.

In [ ]:
ENRICH_KEYS = False   # flip to True to compute keys from audio (slow: loads each file)

if ENRICH_KEYS and n_ready >= 1:
    import librosa
    from src.dsp.mood_engine import analyze_mood
    keys = []
    for _, row in lib.iterrows():
        path = row.get('file_name')
        # file_name may be a bare name; try common locations
        cand = [path, os.path.join('data', str(path))] if path else []
        audio = next((p for p in cand if p and os.path.exists(p)), None)
        if audio is None:
            keys.append(None); continue
        try:
            y, sr = librosa.load(audio, sr=22050, mono=True)
            keys.append(analyze_mood(y, sr).camelot_key)
        except Exception as e:
            print('key failed for', audio, '->', e); keys.append(None)
    lib['camelot_key'] = keys
    print('Enriched keys:', lib['camelot_key'].tolist())
else:
    print('Skipped key enrichment. Harmonic score will be neutral unless camelot_key is present.')

## 4. Scoring — Camelot harmonic rule + compatibility blend

In [ ]:
def camelot_parse(code):
    """'7A' -> (7, 'A'); returns None if unparseable/missing."""
    if code is None or (isinstance(code, float) and np.isnan(code)):
        return None
    m = re.match(r'^\s*(\d{1,2})\s*([ABab])\s*$', str(code))
    return (int(m.group(1)), m.group(2).upper()) if m else None

def harmonic_score(a, b):
    """Camelot-wheel compatibility (0-1). Neutral 0.5 when a key is unknown."""
    pa, pb = camelot_parse(a), camelot_parse(b)
    if pa is None or pb is None:
        return 0.5
    (na, la), (nb, lb) = pa, pb
    dist = min(abs(na - nb), 12 - abs(na - nb))   # distance around the 12-hour wheel
    if na == nb and la == lb:  return 1.00        # identical key — perfect
    if na == nb and la != lb:  return 0.90        # relative major/minor (8A<->8B)
    if dist == 1 and la == lb: return 0.90        # adjacent (+/-1), same mode
    if dist == 2 and la == lb: return 0.70        # +/-2 — energy shift, still ok
    if dist == 1 and la != lb: return 0.60        # diagonal neighbour
    return 0.30                                    # clash-prone

def mood_vec(row):
    d = {k: (0.0 if pd.isna(row.get(k)) else float(row.get(k))) for k in MOOD_DIMS}
    return d if any(v for v in d.values()) else None

def score_pair(a, b):
    bpm = _calculate_bpm_compatibility(a['bpm'], b['bpm'])
    key = harmonic_score(a.get('camelot_key'), b.get('camelot_key'))
    mood = _calculate_mood_continuity(mood_vec(a), mood_vec(b))
    energy = _calculate_energy_arc(a.get('rms_mean'), b.get('rms_mean'))
    overall = (bpm*WEIGHTS['bpm'] + key*WEIGHTS['key'] +
               mood*WEIGHTS['mood'] + energy*WEIGHTS['energy'])
    return dict(bpm=round(bpm,3), key=round(key,3), mood=round(mood,3),
                energy=round(energy,3), overall=round(overall,3))
print('scoring helpers ready')

## 5. Score every pair & rank

In [ ]:
ready = lib[lib['bpm'].notna()].copy()
records = []
for a, b in itertools.combinations(ready.to_dict('records'), 2):
    if BPM_TOLERANCE is not None and abs(a['bpm'] - b['bpm']) > BPM_TOLERANCE:
        continue
    s = score_pair(a, b)
    if s['overall'] < MIN_SCORE:
        continue
    # play the lower-energy track first (build up); fall back to lower BPM
    ra, rb = a.get('rms_mean'), b.get('rms_mean')
    if ra is not None and rb is not None:
        first, second = (a, b) if ra <= rb else (b, a)
    else:
        first, second = (a, b) if a['bpm'] <= b['bpm'] else (b, a)
    def _name(t):
        return t.get('title') or t.get('file_name') or f"id{t['id']}"
    pitch_pct = round((first['bpm'] - second['bpm']) / second['bpm'] * 100, 2)
    records.append({
        'play_first': _name(first), 'then': _name(second),
        'first_id': first['id'], 'second_id': second['id'],
        'first_bpm': round(first['bpm'],2), 'second_bpm': round(second['bpm'],2),
        'bpm_gap': round(abs(a['bpm']-b['bpm']),2),
        'pitch_%_on_2nd': pitch_pct,
        'first_key': first.get('camelot_key'), 'second_key': second.get('camelot_key'),
        **s,
    })

PAIR_COLS = ['play_first','then','first_id','second_id','first_bpm','second_bpm',
             'bpm_gap','pitch_%_on_2nd','first_key','second_key',
             'bpm','key','mood','energy','overall']
pairs = pd.DataFrame(records, columns=PAIR_COLS)
if not pairs.empty:
    pairs = pairs.sort_values('overall', ascending=False).reset_index(drop=True)
print(f'{len(pairs)} candidate pairs (BPM_TOLERANCE={BPM_TOLERANCE}, MIN_SCORE={MIN_SCORE}).')
if pairs.empty:
    print('Nothing to rank yet — analyze at least 2 tracks: python -m src.cli analyze data/')
pairs.head(TOP_N)

## 6. Read the table

- **overall** — blended mix-ability (1.0 = ideal). Aim for pairs above ~0.8.
- **bpm_gap** — raw BPM difference; **pitch_%_on_2nd** is how far you'd pitch the incoming track to beatmatch.
  Most controllers have +/-8% range, so keep |pitch| under 8.
- **key** — 1.0 same key, 0.9 relative/adjacent (the harmonic sweet spot), 0.3 likely to clash.
  Will be ~0.5 everywhere until keys are present (see cell 3).
- **play_first / then** — suggested order (open with the calmer track).

## 7. Mix sheet for one pair

In [ ]:
def segments_for(track_id, db_path=DB_PATH):
    conn = sqlite3.connect(db_path); conn.row_factory = sqlite3.Row
    try:
        return [dict(r) for r in conn.execute(
            'SELECT segment_type, start_time, end_time FROM segments WHERE track_id=? ORDER BY start_time',
            (track_id,))]
    finally:
        conn.close()

def _first_seg(segs, prefix):
    return next((s for s in segs if str(s['segment_type']).lower().startswith(prefix)), None)

def mix_sheet(rank=0):
    if pairs.empty:
        print('No pairs to show.'); return
    p = pairs.iloc[rank]
    out_segs = segments_for(int(p['first_id']))   # outgoing track (playing first)
    in_segs  = segments_for(int(p['second_id']))  # incoming track
    mix_out = _first_seg(out_segs, 'outro') or (out_segs[-1] if out_segs else None)
    mix_in  = _first_seg(in_segs, 'intro')  or (in_segs[0]  if in_segs  else None)
    def mmss(t):
        return '—' if t is None else f'{int(t//60)}:{int(t%60):02d}'
    print(f'🎧  MIX SHEET — pair #{rank}  (overall {p["overall"]})')
    print('=' * 58)
    print(f'PLAY FIRST : {p["play_first"]}  @ {p["first_bpm"]} BPM  key {p["first_key"]}')
    print(f'THEN       : {p["then"]}  @ {p["second_bpm"]} BPM  key {p["second_key"]}')
    print('-' * 58)
    print(f'Beatmatch  : pitch the INCOMING track {p["pitch_%_on_2nd"]:+.2f}%  (gap {p["bpm_gap"]} BPM)')
    warn = '' if abs(p['pitch_%_on_2nd']) <= 8 else '  ⚠️ exceeds typical +/-8% range'
    print(f'             {warn}' if warn else '             within controller pitch range ✓')
    print(f'Harmonic   : {p["first_key"]} -> {p["second_key"]}  (key score {p["key"]})')
    print('-' * 58)
    print(f'Mix OUT of first track near : {mmss(mix_out["start_time"]) if mix_out else "—"}'
          f'  ({mix_out["segment_type"] if mix_out else "no segments"})')
    print(f'Bring IN second track from  : {mmss(mix_in["start_time"]) if mix_in else "0:00"}'
          f'  ({mix_in["segment_type"] if mix_in else "no segments"})')
    print('-' * 58)
    print('EQ tip: cut the incoming track\'s BASS while both play, then swap lows on the drop.')

mix_sheet(0)

## 8. Export the ranked pairs

In [ ]:
os.makedirs('results', exist_ok=True)
out_csv = 'results/mix_pairs.csv'
pairs.to_csv(out_csv, index=False)
print(f'Wrote {len(pairs)} pairs -> {out_csv}')

## Notes & limitations

- **Key isn't stored by the pipeline.** Harmonic scores are neutral (0.5) until you either run the
  optional enrichment (cell 3) or add a `camelot_key` column to the `features` table and persist it
  during analysis. Harmonic mixing is the single biggest lever for clean pairs — worth wiring in.
- **Normalization done right here.** The repo's `find_similar_tracks` z-scores a single vector
  (which collapses to zeros); this notebook scores pairs directly via the transition components instead.
- **`transition_mapper`'s key scorer** expects note-names (`'A'`) but the analyzer emits Camelot
  (`'7A'`), so this notebook uses its own `harmonic_score`. Consider reconciling that in code later.
- **Segments/mix points** depend on the phrasing engine having written `segments`; if a track has none,
  the mix sheet falls back to start/end.

### Suggested next steps
1. Persist `camelot_key` (and maybe key confidence) in `features` during `analyze`.
2. Add a per-pair 'element-onset' aware mix-in point once that detector lands (from the brainstorm).
3. Export the top pairs' cue points into DJUCED/Serato so mix-in/out marks show on the decks.